In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np

#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [2]:
with engine.connect() as connection:
    try:

            query = f"""
            
            SELECT *
                FROM OPENQUERY([172.16.1.175],
'
    SELECT DISTINCT C.NUM_IDENTIFICACION, H.COD_PERIODO, H.DEF_HISTORIA, H.OBS_NOTA, H.COD_MATERIA, b.COD_UNIDAD, b.COD_PENSUM
    FROM SRC_HIS_ACADEMICA H
    LEFT JOIN SINU.SRC_ALUM_PROGRAMA B ON H.ID_ALUM_PROGRAMA = B.ID_ALUM_PROGRAMA
    LEFT JOIN SINU.BAS_TERCERO C ON B.ID_TERCERO = C.ID_TERCERO
    WHERE H.COD_MATERIA IN (''14153'', ''14154'', ''14155'', ''DIS50'', ''DPC12'', ''IDD27'', ''ISD16'', ''SD008'', ''SD012'', ''ADD17'', 
                            ''DPC11'', ''ISD29'', ''SD020'', ''DBM45'', ''DBC37'', ''AED20'', ''AED34'', ''BMD42'', ''DBI41'', ''IED47'', 
                            ''DBG45'', ''BEU41'', ''BMD18'', ''DBV24'', ''DCS24'', ''DTA43'', ''IDD43'', ''DBG46'', ''DBM44'', ''ACD08'', 
                            ''ADD09'', ''DIS49'', ''SD037'', ''DBV23'', ''DBC36'', ''69292'', ''BEU19'', ''BEU29'', ''DBI40'', ''DCS13'', 
                            ''DCS48'', ''DIS48'', ''DTA42'', ''IED31'', ''ACD04'', ''ACD13'', ''DBC38'', ''69290'', ''AED13'', ''BMD28'', 
                            ''ISD45'', ''DBI42'', ''ESP52'', ''14178'', ''14177'', ''ESP22'', ''69291'', ''ESP25'', ''24132'', ''ESP20'', 
                            ''IED31'',''IED47'',''IDD27'',''IDD43'',''DPC13'')
    AND H.ID_ALUM_PROGRAMA IN (
        SELECT P.ID_ALUM_PROGRAMA 
        FROM SRC_ALUM_PROGRAMA P 
        INNER JOIN BAS_TERCERO B ON B.ID_TERCERO = P.ID_TERCERO
    )
')
            """
            resultado = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

### CONSULTA PARA SABER QUE ESTUDIANTES ESTAN VIENDO OPCIÓN DE GRADO

In [3]:
with engine.connect() as connection:
    try:

            query = f"""
            
           SELECT DISTINCT  * 
        
        FROM OPENQUERY([172.16.1.175],
        'SELECT  DISTINCT C.NUM_IDENTIFICACION, A.ID_ALUM_PROGRAMA, A.ID_GRUPO, A.COD_MATERIA, A.COD_PERIODO, D.NOM_MATERIA,A.NOT_DEFINITIVA AS DEF_HISTORIA ,B.COD_UNIDAD,B.COD_PENSUM
        FROM SINU.SRC_ENC_MATRICULA A
        LEFT JOIN SINU.SRC_ALUM_PROGRAMA B ON A.ID_ALUM_PROGRAMA = B.ID_ALUM_PROGRAMA 
        LEFT JOIN SINU.SRC_MATERIA D ON A.COD_MATERIA = D.COD_MATERIA
        LEFT JOIN SINU.BAS_TERCERO C ON B.ID_TERCERO = C.ID_TERCERO
        WHERE D.NOM_MATERIA LIKE ''%OPCI%'' ')
           
            """
            df_faltantes = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

In [4]:
df_faltantes = df_faltantes.sort_values('COD_PERIODO')

In [5]:
df_faltantes = df_faltantes.groupby(['NUM_IDENTIFICACION', 'COD_MATERIA']).last().reset_index()

In [6]:
df_faltantes['NUM_IDENTIFICACION'] = df_faltantes['NUM_IDENTIFICACION'].astype(str)
df_faltantes['COD_MATERIA'] = df_faltantes['COD_MATERIA'].astype(str)

resultado['NUM_IDENTIFICACION'] = resultado['NUM_IDENTIFICACION'].astype(str)
resultado['COD_MATERIA'] = resultado['COD_MATERIA'].astype(str)

df_faltantes['llave'] = df_faltantes['NUM_IDENTIFICACION'] + '_' + df_faltantes['COD_MATERIA']
resultado['llave'] = resultado['NUM_IDENTIFICACION'] + '_' + resultado['COD_MATERIA']

df_concatenar = df_faltantes[~df_faltantes['llave'].isin(resultado['llave'])]

In [7]:
df_concatenar['ESTADO'] = 'CURSANDO'

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_21116\3052513285.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_concatenar['ESTADO'] = 'CURSANDO'


In [8]:
df_op_grado = pd.concat([resultado,df_concatenar],ignore_index=True)

In [9]:
df_op_grado[df_op_grado['NUM_IDENTIFICACION']=='1000457898']

,NUM_IDENTIFICACION,COD_PERIODO,DEF_HISTORIA,OBS_NOTA,COD_MATERIA,COD_UNIDAD,COD_PENSUM,llave,ID_ALUM_PROGRAMA,ID_GRUPO,NOM_MATERIA,ESTADO
275170,1000457898,2025A,4.3,Traslado de notas,14154,1TC11,3028N,1000457898_14154,NaN,NaN,NaN,NaN
276326,1000457898,2024A,3.2,Homologación,14153,1GC11,3028N,1000457898_14153,NaN,NaN,NaN,NaN
651453,1000457898,2024A,3.2,Traslado de notas,14153,1TC11,3028N,1000457898_14153,NaN,NaN,NaN,NaN
656070,1000457898,2026A,4.2,Traslado de notas,14155,1GC11,3028N,1000457898_14155,NaN,NaN,NaN,NaN
698475,1000457898,2023B,2.8,Traslado de notas,14153,1TC11,3028N,1000457898_14153,NaN,NaN,NaN,NaN
753413,1000457898,2024B,2.5,Traslado de notas,14154,1TC11,3028N,1000457898_14154,NaN,NaN,NaN,NaN
806012,1000457898,2025A,4.3,Homologación,14154,1GC11,3028N,1000457898_14154,NaN,NaN,NaN,NaN


In [10]:
# Usamos loc para crear una nueva columna que clasifique si aprobó o reprobó
df_op_grado.loc[df_op_grado['DEF_HISTORIA'] >= 3.0, 'Resultado'] = 'APROBÓ'
df_op_grado.loc[df_op_grado['DEF_HISTORIA'] < 3.0, 'Resultado'] = 'NO APROBADO'

df_op_grado.loc[(df_op_grado['ESTADO']=='CURSANDO'),'Resultado'] = df_op_grado['ESTADO']

In [11]:
materias_ciclo = {
    
    "14153": "TECNICO",
    "14154": "TECNOLOGO",
    "14155": "PROFESIONAL",
    "DIS50": "PROFESIONAL",
    "DPC12": "TECNOLOGO",
    "IDD27": "TECNOLOGO",
    "ISD16": "TECNICO",
    "SD008": "TECNICO",
    "SD012": "PROFESIONAL",
    "ADD17": "TECNOLOGO",
    "DPC11": "TECNICO",
    "ISD29": "TECNOLOGO",
    "SD020": "TECNOLOGO",
    "DBM45": "TECNOLOGO",
    "DBC37": "TECNOLOGO",
    "AED20": "TECNOLOGO",
    "AED34": "PROFESIONAL",
    "BMD42": "PROFESIONAL",
    "DBI41": "TECNOLOGO",
    "IED47": "PROFESIONAL",
    "DBG45": "TECNICO",
    "BEU41": "PROFESIONAL",
    "BMD18": "TECNICO",
    "DBV24": "TECNOLOGO",
    "DCS24": "TECNOLOGO",
    "DTA43": "TECNOLOGO",
    "IDD43": "PROFESIONAL",
    "DBG46": "TECNOLOGO",
    "DBM44": "TECNICO",
    "ACD08": "TECNOLOGO",
    "ADD09": "TECNICO",
    "DIS49": "TECNOLOGO",
    "SD037": "PROFESIONAL",
    "DBV23": "TECNICO",
    "DBC36": "TECNICO",
    "69292": "PROFESIONAL",
    "BEU19": "TECNICO",
    "BEU29": "TECNOLOGO",
    "DBI40": "TECNICO",
    "DCS13": "TECNICO",
    "DCS48": "PROFESIONAL",
    "DIS48": "TECNICO",
    "DTA42": "TECNICO",
    "IED31": "TECNOLOGO",
    "ACD04": "TECNICO",
    "ACD13": "PROFESIONAL",
    "DBC38": "PROFESIONAL",
    "69290": "TECNICO",
    "AED13": "TECNICO",
    "BMD28": "TECNOLOGO",
    "ISD45": "PROFESIONAL",
    "DBI42": "PROFESIONAL",
    "ESP52": "SEMILLERO",
    "14178": "OPCION DE GRADO",
    "14177": "OPCION DE GRADO",
    "ESP22": "TECNOLOGO",
    "69291": "TECNOLOGO",
    "ESP25": "PROFESIONAL",
    "24132": "TECNOLOGO",
    "ESP20": "TECNICO",
    "IED41": "OPCION DE GRADO",
    "IED31 ":"TECNOLOGO",
    "IED47":"PROFESIONAL",
    "IDD27 ":"TECNOLOGO",
    "IDD43":"PROFESIONAL",
    "DPC13":"PROFESIONAL",
    "DBM46":"PROFESIONAL",
    "DBG47":"PROFESIONAL",
    "DTA44":"PROFESIONAL"
   
   
    
}

df_op_grado['NIVEL'] = df_op_grado['COD_MATERIA'].map(materias_ciclo)

In [12]:
df_final = df_op_grado[['NUM_IDENTIFICACION','COD_PERIODO', 'DEF_HISTORIA','COD_MATERIA','Resultado', 'NIVEL','COD_UNIDAD','COD_PENSUM']]

In [13]:
df_final2 =  df_final.drop_duplicates()

In [14]:
# Ordenar el DataFrame para que "APROBÓ" tenga prioridad
df_sorted = df_final2.sort_values(by=['Resultado'], ascending=False, key=lambda col: col == "APROBÓ")

# Eliminar duplicados priorizando "APROBÓ"
df_priorizado = df_sorted.drop_duplicates(subset=['NUM_IDENTIFICACION', 'COD_MATERIA','NIVEL'])

In [15]:
df_priorizado[df_priorizado['NUM_IDENTIFICACION']=='1000457898']

,NUM_IDENTIFICACION,COD_PERIODO,DEF_HISTORIA,COD_MATERIA,Resultado,NIVEL,COD_UNIDAD,COD_PENSUM
806012,1000457898,2025A,4.3,14154,APROBÓ,TECNOLOGO,1GC11,3028N
656070,1000457898,2026A,4.2,14155,APROBÓ,PROFESIONAL,1GC11,3028N
651453,1000457898,2024A,3.2,14153,APROBÓ,TECNICO,1TC11,3028N


In [16]:
df_priorizado['COD_PENSUM'] = df_priorizado['COD_PENSUM'].str.rstrip('ND')

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_21116\3978895575.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_priorizado['COD_PENSUM'] = df_priorizado['COD_PENSUM'].str.rstrip('ND')


In [17]:
df_priorizado.shape

(371199, 8)

### Identificar los estudiantes NO APROBADO comunes con los que estan CURSANDO

In [18]:
df_op_grado_no_aprobados = df_priorizado[df_priorizado['Resultado']=='NO APROBADO']

In [19]:
df_faltantes['NUM_IDENTIFICACION'] = df_faltantes['NUM_IDENTIFICACION'].astype(str)
df_faltantes['COD_MATERIA'] = df_faltantes['COD_MATERIA'].astype(str)

df_op_grado_no_aprobados['NUM_IDENTIFICACION'] = df_op_grado_no_aprobados['NUM_IDENTIFICACION'].astype(str)
df_op_grado_no_aprobados['COD_MATERIA'] = df_op_grado_no_aprobados['COD_MATERIA'].astype(str)

df_faltantes['llave'] = df_faltantes['NUM_IDENTIFICACION'] + '_' + df_faltantes['COD_MATERIA']
df_op_grado_no_aprobados['llave'] = df_op_grado_no_aprobados['NUM_IDENTIFICACION'] + '_' + df_op_grado_no_aprobados['COD_MATERIA']

df_comunes = df_faltantes[df_faltantes['llave'].isin(df_op_grado_no_aprobados['llave'])]

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_21116\930014477.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_op_grado_no_aprobados['NUM_IDENTIFICACION'] = df_op_grado_no_aprobados['NUM_IDENTIFICACION'].astype(str)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_21116\930014477.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_op_grado_no_aprobados['COD_MATERIA'] = df_op_grado_no_aprobados['COD_MATERIA'].astype(str)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_

In [20]:
df_priorizado[df_priorizado['NUM_IDENTIFICACION']=='1000457898']

,NUM_IDENTIFICACION,COD_PERIODO,DEF_HISTORIA,COD_MATERIA,Resultado,NIVEL,COD_UNIDAD,COD_PENSUM
806012,1000457898,2025A,4.3,14154,APROBÓ,TECNOLOGO,1GC11,3028
656070,1000457898,2026A,4.2,14155,APROBÓ,PROFESIONAL,1GC11,3028
651453,1000457898,2024A,3.2,14153,APROBÓ,TECNICO,1TC11,3028


In [21]:
df_comunes[df_comunes['NUM_IDENTIFICACION']=='1000457898']

,NUM_IDENTIFICACION,COD_MATERIA,ID_ALUM_PROGRAMA,ID_GRUPO,COD_PERIODO,NOM_MATERIA,DEF_HISTORIA,COD_UNIDAD,COD_PENSUM,llave


In [22]:
# Hacer el merge para encontrar los registros en común
df_priorizado = df_priorizado.merge(df_comunes[['NUM_IDENTIFICACION', 'COD_MATERIA', 'COD_PERIODO']], 
                                    on=['NUM_IDENTIFICACION', 'COD_MATERIA'], 
                                    how='left', 
                                    suffixes=('', '_nuevo'))

# Actualizar el Estado a "Cursando" donde haya coincidencia
df_priorizado.loc[df_priorizado['COD_PERIODO_nuevo'].notna(), 'Resultado'] = 'CURSANDO'

# Actualizar el PERIODO con el PERIODO de df_comunes
df_priorizado.loc[df_priorizado['COD_PERIODO_nuevo'].notna(), 'COD_PERIODO'] = df_priorizado['COD_PERIODO_nuevo']

# Eliminar la columna auxiliar COD_PERIODO_nuevo
df_priorizado.drop(columns=['COD_PERIODO_nuevo'], inplace=True)


In [23]:
df_priorizado[df_priorizado['NUM_IDENTIFICACION']=='1000579721']

,NUM_IDENTIFICACION,COD_PERIODO,DEF_HISTORIA,COD_MATERIA,Resultado,NIVEL,COD_UNIDAD,COD_PENSUM
167361,1000579721,2025C,3.4,14155,APROBÓ,PROFESIONAL,1PC11,3028
271510,1000579721,2023A,4.5,14153,APROBÓ,TECNICO,1PC11,3028
283717,1000579721,2023A,4.5,14154,APROBÓ,TECNOLOGO,1PC11,3028


In [24]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'OPCION_GRADO_HISTORIA_OP'
schema_name = 'SRUBIO'

# Send the data to sql server
df_priorizado.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

### CONSULTA ESTUDIANTES UNICOS

In [25]:
with engine.connect() as connection:
    try:

            query = f"""
            
            select DISTINCT NUM_IDENTIFICACION from Estudiantes_activos
            
            """
            df_est_uni = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

##### SACO DE LA TABLA ESTUDIANTES ACTIVOS LOS QUE YA EXISTEN EN OPCIÓN DE GRADO

In [26]:
df_est_uni['NUM_IDENTIFICACION'].astype(str)

df_distintos = df_est_uni[~df_est_uni['NUM_IDENTIFICACION'].isin(df_priorizado['NUM_IDENTIFICACION'])]

In [27]:
df_distintos['NUM_IDENTIFICACION'].nunique()

58642

##### ME QUEDO CON LAS CEDULAS DE LOS ESTUDIANTES ACTIVOS

In [28]:
df_priorizado_acti = df_priorizado[df_priorizado['NUM_IDENTIFICACION'].isin(df_est_uni['NUM_IDENTIFICACION'])]

In [29]:
df_priorizado_acti['NUM_IDENTIFICACION'].nunique()

38694

In [30]:
df_distintos['Resultado'] = 'SIN CURSAR'

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_21116\3843064859.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_distintos['Resultado'] = 'SIN CURSAR'


In [31]:
df_final_estudiantes_activos = pd.concat([df_priorizado_acti,df_distintos],ignore_index=True)

In [32]:
df_final_estudiantes_activos['NUM_IDENTIFICACION'].nunique()

97336

In [33]:
df_final_estudiantes_activos['COD_PENSUM'] = df_final_estudiantes_activos['COD_PENSUM'].str.rstrip('ND')

In [34]:
df_final_estudiantes_activos.shape

(138928, 8)

In [35]:
df_final_estudiantes_activos = df_final_estudiantes_activos.sort_values(['COD_PERIODO'])

In [36]:
df_final_estudiantes_activos['NUM_IDENTIFICACION'].nunique()

97336

In [37]:
df_final_estudiantes_activos = df_final_estudiantes_activos.groupby(
    ['NUM_IDENTIFICACION', 'NIVEL'], dropna=False
).tail(1)

In [38]:
df_final_estudiantes_activos['NUM_IDENTIFICACION'].nunique()

97336

In [39]:
df_final_estudiantes_activos[df_final_estudiantes_activos['NUM_IDENTIFICACION']=='1000579721']

,NUM_IDENTIFICACION,COD_PERIODO,DEF_HISTORIA,COD_MATERIA,Resultado,NIVEL,COD_UNIDAD,COD_PENSUM


In [40]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'OPCION_GRADO_ESTUDIANTES_ACTIVOS'
schema_name = 'SRUBIO'

# Send the data to sql server
df_final_estudiantes_activos.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()